# API du modèle

L'API Flask charge, au démarrage, le modèle et `transform_pipeline.pkl` depuis le Model Registry, puis expose `POST /predict`.
Les prédictions `int64` ne sont pas sérialisables en JSON : les convertir en `int`.
Prérequis cloud : VM MLflow, bucket d'artifacts, clé JSON dans `conf/key.json`. Ne pas committer cette clé, ni `.env`.
Variables requises : `ENV`, `MLFLOW_SERVER`, `MLFLOW_REGISTRY_NAME`, `GOOGLE_APPLICATION_CREDENTIALS`.


## Encodage partagé avec Kedro

`encode_features` doit renvoyer les `LabelEncoder` utilisés à l'entraînement. L'API applique les mêmes transformations.
Mettre à jour le test et la fixture en même temps que la valeur de retour. Relancer ensuite `processing`, puis `training`.


Entrée de `catalog.yml`.


Fonction `encode_features` dans `nodes.py`.


Pipeline `processing`.


Dans `auto_ml`, journaliser aussi le pickle d'encodage.


## Projet purchase_predict_api

Dossier distinct de Kedro : `app.py` (routes), `requirements.txt`, `src/` (chargement du modèle).
Réutiliser le compte de service Kedro : nouvelle clé dans `conf/key.json`, chemin déclaré dans `.env`.


Fichier `.env`.


Fichier `requirements.txt`.


Fichier `src/__init__.py`. L'import échoue si une variable manque.


Fichier `src/model.py`. Le modèle chargé est la dernière version dont l'état correspond à `ENV`.


Fichier `app.py`. `python app.py` écoute le port 5000.
Le corps JSON est lu tel quel par `pd.read_json`. La conversion en `int` évite l'erreur de sérialisation des `int64`.


## Test local

Charger `primary.csv` avant encodage : l'API encode elle-même. Lancer `python app.py` dans le terminal du projet, puis exécuter la requête.


In [ ]:
import os
import pandas as pd

dataset = pd.read_csv(os.path.expanduser("data/primary.csv"))
dataset = dataset.drop(["user_session", "user_id", "purchased"], axis=1)


In [ ]:
import requests

requests.post(
    "http://127.0.0.1:5000/predict",
    json=dataset.sample(n=10).to_json()
).json()


## Gunicorn

Flask traite une requête à la fois. Gunicorn lance plusieurs workers et un processus maître qui répartit les requêtes et relance un worker arrêté.
Le processus `grep` affiché par `ps` est la commande de recherche, pas un worker. `-D` détache le serveur. Le port d'écoute reste 5000 en local.


In [ ]:
requests.post(
    "http://127.0.0.1:5000/predict",
    json=dataset.sample(n=10).to_json()
).json()


## Dépôt

`.gitignore` exclut l'environnement local et la clé. Renseigner l'adresse SSH de votre dépôt à la place de `<ADRESSE_SSH>`.
